# 3 — Gráficos: Perfil dos Estudantes

| Código | Tipo | Descrição |
|--------|------|-----------|
| 1 | Pizza (donut) | Distribuição por sexo |
| 2 | Barras horizontais | Distribuição por cor/raça |
| 3 | Barras | Distribuição por faixa etária |
| 4 | Barras | Distribuição por renda familiar |
| 5 | Pizza (donut) | Distribuição por turno |
| 6 | Barras 100% empilhadas | Composição por cor/raça ao longo dos anos |
| 7 | Barras agrupadas | Taxa de evasão por sexo e tipo de curso |
| 8 | Barras | Taxa de evasão por faixa etária |

### 3.1. Importações e Configurações

In [ ]:
import pandas as pd
import plotly.express as px

# Ordem cronológica das faixas etárias (usada para ordenar os eixos)
ORDEM_ETARIA = [
    "Menor de 14 anos", "15 a 19 anos", "20 a 24 anos",
    "25 a 29 anos", "30 a 34 anos", "35 a 39 anos",
    "40 a 44 anos", "45 a 49 anos", "50 a 54 anos",
    "55 a 59 anos", "Maior de 60 anos",
]

# Ordem lógica das faixas de renda
ORDEM_RENDA = [
    "0<RFP<=0,5", "0,5<RFP<=1", "1<RFP<=1,5",
    "1,5<RFP<=2,5", "2,5<RFP<=3,5", "RFP>3,5", "Não declarada",
]

print("Imports OK")

### 3.2. Carga dos dados

In [ ]:
df = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

# Mantém apenas as faixas etárias que existem nos dados
faixas_presentes = [f for f in ORDEM_ETARIA if f in df["Faixa Etária"].unique()]
rendas_presentes = [r for r in ORDEM_RENDA  if r in df["Renda Familiar"].unique()]

print("Shape:", df.shape)
print("Faixas etárias:", faixas_presentes)

### 3.3. Gráficos de distribuição

In [ ]:
# 1: Distribuição por sexo

sexo_df = df["Sexo"].value_counts().reset_index()
sexo_df.columns = ["Sexo", "Qtd"]

fig_g19 = px.pie(
    sexo_df,
    names="Sexo",
    values="Qtd",
    hole=0.4,
    color_discrete_sequence=["#2196F3", "#E91E63", "#9E9E9E"],
    title="G19 — Distribuição por Sexo (2017–2024)",
)
fig_g19.update_traces(textposition="outside", textinfo="percent+label")
fig_g19.show()

In [ ]:
# 2: Distribuição por cor/raça

raca_df = df["Cor / Raça"].value_counts().reset_index()
raca_df.columns = ["Cor/Raça", "Qtd"]

fig_g20 = px.bar(
    raca_df.sort_values("Qtd", ascending=True),
    x="Qtd",
    y="Cor/Raça",
    orientation="h",
    color="Qtd",
    color_continuous_scale="Purples",
    text_auto=True,
    title="G20 — Distribuição por Cor/Raça (2017–2024)",
    labels={"Qtd": "Matrículas"},
)
fig_g20.update_layout(coloraxis_showscale=False)
fig_g20.show()

In [ ]:
# 3: Distribuição por faixa etária
# reindex() garante que as barras apareçam na ordem cronológica correta

etaria_df = (
    df["Faixa Etária"]
    .value_counts()
    .reindex(faixas_presentes)
    .reset_index()
)
etaria_df.columns = ["Faixa Etária", "Qtd"]

fig_g21 = px.bar(
    etaria_df,
    x="Faixa Etária",
    y="Qtd",
    color="Qtd",
    color_continuous_scale="Blues",
    text_auto=True,
    title="G21 — Distribuição por Faixa Etária (2017–2024)",
    labels={"Qtd": "Matrículas"},
)
fig_g21.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30)
fig_g21.show()

In [ ]:
# 4: Distribuição por renda familiar

renda_df = (
    df["Renda Familiar"]
    .value_counts()
    .reindex(rendas_presentes)
    .reset_index()
)
renda_df.columns = ["Renda", "Qtd"]

fig_g22 = px.bar(
    renda_df,
    x="Renda",
    y="Qtd",
    color="Qtd",
    color_continuous_scale="Greens",
    text_auto=True,
    title="G22 — Distribuição por Renda Familiar per capita (salários mínimos)",
    labels={"Qtd": "Matrículas"},
)
fig_g22.update_layout(coloraxis_showscale=False, xaxis_tickangle=-20)
fig_g22.show()

In [ ]:
# 5: Distribuição por turno

turno_df = df["Turno"].value_counts().reset_index()
turno_df.columns = ["Turno", "Qtd"]

fig_g23 = px.pie(
    turno_df,
    names="Turno",
    values="Qtd",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="G23 — Distribuição por Turno (2017–2024)",
)
fig_g23.update_traces(textposition="outside", textinfo="percent+label")
fig_g23.show()

In [ ]:
# 6: Composição por cor/raça ao longo dos anos

raca_ano = (
    df.groupby(["Ano", "Cor / Raça"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g24 = px.bar(
    raca_ano,
    x="Ano",
    y="Qtd",
    color="Cor / Raça",
    barmode="stack",
    title="G24 — Composição por Cor/Raça ao Longo dos Anos (%)",
    labels={"Qtd": "(%)", "Cor / Raça": "Cor/Raça"},
)
fig_g24.update_xaxes(tickmode="linear", dtick=1)
fig_g24.show()

### 3.4 Gráficos de evasão por perfil

In [ ]:
# 7: Taxa de evasão por sexo e tipo de curso (barras agrupadas)
# apply() permite calcular uma métrica personalizada por grupo

evad_sexo_tipo = (
    df.groupby(["Tipo de Curso", "Sexo"])
    .apply(
        lambda grupo: pd.Series({
            "TE": (
                (grupo["Categoria da Situação"] == "Evadidos").sum()
                / max(len(grupo), 1)
                * 100
            )
        })
    )
    .reset_index()
)

fig_g25 = px.bar(
    evad_sexo_tipo,
    x="Tipo de Curso",
    y="TE",
    color="Sexo",
    barmode="group",
    title="G25 — Taxa de Evasão por Sexo e Tipo de Curso",
    labels={"TE": "Taxa de Evasão (%)", "Tipo de Curso": ""},
)
fig_g25.update_layout(xaxis_tickangle=-20)
fig_g25.show()

In [ ]:
# 8: Taxa de evasão por faixa etária

evad_etaria = (
    df.groupby("Faixa Etária")
    .apply(
        lambda grupo: pd.Series({
            "TE": (
                (grupo["Categoria da Situação"] == "Evadidos").sum()
                / max(len(grupo), 1)
                * 100
            )
        })
    )
    .reset_index()
)

# Ordena pelas faixas etárias na ordem cronológica
evad_etaria["Faixa Etária"] = pd.Categorical(
    evad_etaria["Faixa Etária"],
    categories=faixas_presentes,
    ordered=True,
)
evad_etaria = evad_etaria.sort_values("Faixa Etária")

fig_g26 = px.bar(
    evad_etaria,
    x="Faixa Etária",
    y="TE",
    color="TE",
    color_continuous_scale="Oranges",
    text_auto=".1f",
    title="G26 — Taxa de Evasão por Faixa Etária (2017–2024)",
    labels={"TE": "Taxa de Evasão (%)"},
)
fig_g26.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30)
fig_g26.show()